# SIM Intervention Demand Forecasting — Model Training

## 5.1. Objective

The objective of this stage is to train and evaluate machine learning models for forecasting daily SIM intervention counts across six operational divisions.

We will begin with LightGBM and then evaluate CatBoost using the same chronological validation framework.

The models will use the 12 candidate predictive features developed in the Feature Engineering stage.

Model evaluation will follow these requirements:

- Use monthly expanding-window validation from January 2021 through December 2024.
- Respect the D−3 historical information cutoff.
- Keep the model fixed within each validation month.
- Compare predictions using Mean Absolute Error (MAE) and prediction bias.
- Use the selected 13-observation same-weekday baseline as the reference for comparison.
- Keep the 2025–2026 final test period reserved.

The goal is to determine whether the machine learning models improve forecasting accuracy compared with the baseline, while maintaining a reproducible and chronologically valid evaluation process.

## 5.2. Install and Import Required Libraries

We will install LightGBM and import the Python libraries required for data preparation and model training.

We will use pandas and NumPy to work with the datasets, LightGBM to train the forecasting model, and scikit-learn to calculate evaluation metrics.

CatBoost will be introduced later, after the initial LightGBM implementation has been verified.

In [ ]:
# LightGBM and CatBoost are project dependencies (pyproject.toml): no install needed in the repo.

In [ ]:

# Import libraries
import pandas as pd
import numpy as np

import lightgbm as lgb

from sklearn.metrics import mean_absolute_error

# Verify the LightGBM installation
print("LightGBM version:", lgb.__version__)


LightGBM version: 4.7.0


## 5.3. Load Modeling Data and Validation Schedule

We will load the supervised modeling dataset and the chronological validation schedule generated in previous stages.

The modeling dataset provides the predictive features and known intervention counts. The validation schedule defines the monthly training cutoffs and evaluation periods.

We will verify both inputs before training the first LightGBM model.

In [ ]:

# Define file paths
modeling_file = "../../data/interim/modelling/SIM_Features_Modeling.csv"
schedule_file = "../../data/interim/modelling/SIM_Chronological_Validation_Schedule.csv"

# Load the modeling dataset
df_modeling = pd.read_csv(
    modeling_file,
    parse_dates=["Date"]
)

# Load the chronological validation schedule
df_schedule = pd.read_csv(
    schedule_file,
    parse_dates=[
        "Validation_Start",
        "Validation_End",
        "Training_Cutoff"
    ]
)

# Sort modeling observations chronologically
df_modeling = df_modeling.sort_values(
    ["Date", "DIVISION"]
).reset_index(drop=True)

# Verify the datasets
print("Modeling dataset dimensions:", df_modeling.shape)
print("Validation schedule dimensions:", df_schedule.shape)

print("Modeling start date:", df_modeling["Date"].min().date())
print("Modeling end date:", df_modeling["Date"].max().date())

print("Validation months:", len(df_schedule))

# Preview the validation schedule
display(df_schedule.head())


Modeling dataset dimensions: (14748, 14)
Validation schedule dimensions: (48, 5)
Modeling start date: 2020-01-01
Modeling end date: 2026-09-24
Validation months: 48


,Validation_Start,Validation_End,Training_Cutoff,Training_Rows,Validation_Rows
0,2021-01-01,2021-01-31,2020-12-29,2184,186
1,2021-02-01,2021-02-28,2021-01-29,2370,168
2,2021-03-01,2021-03-31,2021-02-26,2538,186
3,2021-04-01,2021-04-30,2021-03-29,2724,180
4,2021-05-01,2021-05-31,2021-04-28,2904,186


## 5.4. Define Predictive Features and Target

We will identify the input features (X) and target variable (y) used to train the machine learning models.

The models will use 12 candidate predictive features: the operational division, five calendar features, four lag features, and two rolling average features.

The target variable is `Intervention_Count`, representing the recorded daily interventions for each operational division.

The `Date` column will be used to control chronological training and validation periods, but it will not be included directly as a predictive feature.

In [ ]:

# Define the predictive features
feature_columns = [
    "DIVISION",
    "DayOfWeek",
    "Month",
    "DayOfMonth",
    "IsWeekend",
    "DayOfYear",
    "Lag_3",
    "Lag_7",
    "Lag_14",
    "Lag_28",
    "Rolling_Mean_7",
    "Rolling_Mean_14"
]

# Define the target variable
target_column = "Intervention_Count"

# Verify the selected columns
print("Number of predictive features:", len(feature_columns))
print("Target variable:", target_column)

print("\nPredictive features:")
for column in feature_columns:
    print("-", column)


Number of predictive features: 12
Target variable: Intervention_Count

Predictive features:
- DIVISION
- DayOfWeek
- Month
- DayOfMonth
- IsWeekend
- DayOfYear
- Lag_3
- Lag_7
- Lag_14
- Lag_28
- Rolling_Mean_7
- Rolling_Mean_14


## 5.5. Prepare the First Training and Validation Window

We will use January 2021 as the first example to understand the model training process.

The training dataset will include observations through December 29, 2020, following the D−3 cutoff. The validation dataset will contain observations from January 1 through January 31, 2021.

We will separate predictive features (X) from known intervention counts (y).

The validation target values will be used only to evaluate the predictions, not to train the model.

In [ ]:

# Select the first validation window
first_window = df_schedule.iloc[0]

# Define the training and validation periods
training_cutoff = first_window["Training_Cutoff"]
validation_start = first_window["Validation_Start"]
validation_end = first_window["Validation_End"]

# Select training observations
df_train = df_modeling[
    df_modeling["Date"] <= training_cutoff
].copy()

# Select validation observations
df_val = df_modeling[
    (df_modeling["Date"] >= validation_start) &
    (df_modeling["Date"] <= validation_end)
].copy()

# Separate features and target
X_train = df_train[feature_columns]
y_train = df_train[target_column]

X_val = df_val[feature_columns]
y_val = df_val[target_column]

# Verify the results
print("Training cutoff:", training_cutoff.date())
print("Validation period:", validation_start.date(), "to", validation_end.date())

print("\nX_train dimensions:", X_train.shape)
print("y_train observations:", len(y_train))

print("X_val dimensions:", X_val.shape)
print("y_val observations:", len(y_val))

print("\nMissing training targets:", y_train.isna().sum())
print("Missing validation targets:", y_val.isna().sum())


Training cutoff: 2020-12-29
Validation period: 2021-01-01 to 2021-01-31

X_train dimensions: (2184, 12)
y_train observations: 2184
X_val dimensions: (186, 12)
y_val observations: 186

Missing training targets: 0
Missing validation targets: 0


## 5.6. Train the First LightGBM Model

We will train an initial LightGBM regression model using historical observations available through December 29, 2020.

The model will learn relationships between the 12 candidate predictive features and daily intervention counts.

We will use a simple initial configuration without hyperparameter tuning. This first model is intended to demonstrate the training and prediction process before applying the monthly validation framework.

The model will be evaluated on January 2021 observations that were not used during training.

In [ ]:

# Create the LightGBM regression model
model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=15,
    random_state=42,
    verbosity=-1
)

# Train the model using historical data
model.fit(X_train, y_train)

# Generate predictions for January 2021
y_pred = model.predict(X_val)

# Display the first predictions
print("Model training completed.")
print("Number of predictions:", len(y_pred))

# Compare actual and predicted counts
df_predictions = df_val[
    ["Date", "DIVISION", "Intervention_Count"]
].copy()

df_predictions["Predicted_Count"] = y_pred

display(df_predictions.head(10))


Model training completed.
Number of predictions: 186


,Date,DIVISION,Intervention_Count,Predicted_Count
2196,2021-01-01,1,18.0,23.121406
2197,2021-01-01,2,41.0,54.634131
2198,2021-01-01,3,48.0,58.526979
2199,2021-01-01,4,53.0,58.386332
2200,2021-01-01,5,42.0,58.798730
2201,2021-01-01,6,27.0,21.365129
2202,2021-01-02,1,22.0,22.459365
2203,2021-01-02,2,35.0,53.405138
2204,2021-01-02,3,48.0,59.764484
2205,2021-01-02,4,48.0,60.178477


## 5.7. Evaluate the First LightGBM Model

We will evaluate the initial LightGBM model on January 2021 observations.

Mean Absolute Error (MAE) will measure the average magnitude of prediction errors. Prediction bias will indicate whether the model tends to overestimate or underestimate intervention counts.

These results will provide an initial example of model performance. They will not be used as the final validation result or as evidence that LightGBM outperforms the baseline across the full development period.

In [ ]:

# Calculate prediction errors
df_predictions["Error"] = (
    df_predictions["Predicted_Count"] -
    df_predictions["Intervention_Count"]
)

# Calculate evaluation metrics
mae = mean_absolute_error(
    df_predictions["Intervention_Count"],
    df_predictions["Predicted_Count"]
)

bias = df_predictions["Error"].mean()

# Display the results
print("Validation period: January 2021")
print("LightGBM MAE:", round(mae, 3))
print("LightGBM Bias:", round(bias, 3))

# Calculate MAE for each operational division
division_mae = (
    df_predictions.groupby("DIVISION")["Error"]
    .apply(lambda errors: errors.abs().mean())
)

print("\nMAE by division:")
display(division_mae)


Validation period: January 2021
LightGBM MAE: 7.781
LightGBM Bias: 2.509

MAE by division:


DIVISION
1     5.340960
2     7.614187
3     7.342713
4     8.507026
5    12.636523
6     5.244993
Name: Error, dtype: float64

**Methodological Observation**

The initial LightGBM model was trained using 2,184 historical observations available through December 29, 2020, and evaluated on 186 observations from January 2021.

The model achieved an MAE of 7.781 interventions and a positive prediction bias of 2.509, indicating an average tendency to overestimate intervention counts during this month.

Prediction accuracy varied across operational divisions. Division 6 had the lowest MAE (5.245), while Division 5 had the highest (12.637).

These results provide an initial assessment of LightGBM performance. They do not establish its overall forecasting accuracy or superiority over the baseline, since only one validation month has been evaluated.

## 5.8. Train LightGBM Across Monthly Validation Windows

We will apply the initial LightGBM configuration to all 48 monthly validation windows from January 2021 through December 2024.

For each month, a new model will be trained using historical observations available through the corresponding D−3 training cutoff. The model will remain fixed throughout the validation month.

Daily validation features may incorporate newly available historical intervention counts, provided that they respect the D−3 information cutoff.

We will store all out-of-sample predictions for subsequent evaluation against the selected baseline. The reserved final test period will remain excluded.

In [ ]:

# Store predictions from all validation months
all_predictions = []

# Repeat training and prediction for each month
for _, window in df_schedule.iterrows():

    # Select training observations
    monthly_train = df_modeling[
        df_modeling["Date"] <= window["Training_Cutoff"]
    ]

    # Select validation observations
    monthly_val = df_modeling[
        (df_modeling["Date"] >= window["Validation_Start"]) &
        (df_modeling["Date"] <= window["Validation_End"])
    ]

    # Prepare features and target
    X_train_month = monthly_train[feature_columns]
    y_train_month = monthly_train[target_column]

    X_val_month = monthly_val[feature_columns]

    # Create a new model for this month
    monthly_model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=15,
        random_state=42,
        verbosity=-1
    )

    # Train the monthly model
    monthly_model.fit(X_train_month, y_train_month)

    # Generate validation predictions
    monthly_pred = monthly_model.predict(X_val_month)

    # Store predictions and actual counts
    monthly_results = monthly_val[
        ["Date", "DIVISION", "Intervention_Count"]
    ].copy()

    monthly_results["Predicted_Count"] = monthly_pred

    all_predictions.append(monthly_results)

# Combine monthly predictions
df_lightgbm_predictions = pd.concat(
    all_predictions,
    ignore_index=True
)

# Verify results
print("Validation months processed:", len(all_predictions))
print("Total predictions:", len(df_lightgbm_predictions))
print("Missing predictions:", df_lightgbm_predictions["Predicted_Count"].isna().sum())

print("\nFirst five predictions:")
display(df_lightgbm_predictions.head())


Validation months processed: 48
Total predictions: 8760
Missing predictions: 0

First five predictions:


,Date,DIVISION,Intervention_Count,Predicted_Count
0,2021-01-01,1,18.0,23.121406
1,2021-01-01,2,41.0,54.634131
2,2021-01-01,3,48.0,58.526979
3,2021-01-01,4,53.0,58.386332
4,2021-01-01,5,42.0,58.798730


## 5.9. Evaluate LightGBM Validation Performance

We will evaluate the LightGBM predictions generated across the 48 monthly validation windows from 2021 through 2024.

Mean Absolute Error (MAE) will measure the average magnitude of prediction errors, while prediction bias will indicate whether the model tends to overestimate or underestimate intervention counts.

We will also examine MAE by operational division and by calendar year to identify differences in forecasting performance.

These results will be compared with the selected baseline using the same validation observations. The reserved final test period will remain excluded.

In [ ]:

# Calculate prediction errors
df_lightgbm_predictions["Error"] = (
    df_lightgbm_predictions["Predicted_Count"] -
    df_lightgbm_predictions["Intervention_Count"]
)

# Calculate the overall validation metrics
overall_mae = df_lightgbm_predictions["Error"].abs().mean()
overall_bias = df_lightgbm_predictions["Error"].mean()

print("LightGBM Validation Results (2021-2024)")
print("Total predictions:", len(df_lightgbm_predictions))
print("Overall MAE:", round(overall_mae, 3))
print("Overall Bias:", round(overall_bias, 3))

# Calculate performance by division
division_metrics = (
    df_lightgbm_predictions.groupby("DIVISION")["Error"]
    .agg(
        MAE=lambda errors: errors.abs().mean(),
        Bias="mean"
    )
)

print("\nPerformance by division:")
display(division_metrics.round(3))

# Extract the year from each forecast date
df_lightgbm_predictions["Year"] = (
    df_lightgbm_predictions["Date"].dt.year
)

# Calculate performance by year
yearly_metrics = (
    df_lightgbm_predictions.groupby("Year")["Error"]
    .agg(
        MAE=lambda errors: errors.abs().mean(),
        Bias="mean"
    )
)

print("\nPerformance by year:")
display(yearly_metrics.round(3))


LightGBM Validation Results (2021-2024)
Total predictions: 8760
Overall MAE: 8.15
Overall Bias: -1.913

Performance by division:


,MAE,Bias
DIVISION,,
1,6.035,-1.351
2,8.255,-1.496
3,9.442,-3.407
4,8.679,-2.200
5,8.846,-1.798
6,7.644,-1.227



Performance by year:


,MAE,Bias
Year,,
2021,7.191,-0.778
2022,7.671,-2.370
2023,9.784,-3.530
2024,7.954,-0.974


**Methodological Observation**

LightGBM was evaluated across 48 monthly validation windows covering 8,760 division-day observations from 2021 through 2024.

The model achieved an overall MAE of 8.150 interventions and a prediction bias of −1.913, indicating a general tendency to underestimate intervention counts.

Prediction performance varied by operational division. Division 1 had the lowest MAE (6.035), while Division 3 had the highest MAE (9.442) and the strongest negative bias (−3.407).

Annual performance also varied, with the highest MAE (9.784) observed in 2023.

These results suggest that LightGBM may improve absolute forecasting accuracy compared with the selected baseline. However, a direct comparison using matched validation observations is required before drawing a conclusion. The model's negative prediction bias also requires consideration.

## 5.10. Load Baseline Validation Predictions

We will load the baseline predictions generated during the Baseline Forecasting stage.

The selected baseline uses the average of the 13 most recent available observations from the same day of the week.

Before comparing the baseline with LightGBM, we will inspect the baseline dataset to identify its columns, number of observations, and date range.

The comparison will use matching dates and operational divisions to ensure that both forecasting methods are evaluated fairly.

In [ ]:

# Define the baseline prediction file
baseline_file = "../../data/interim/modelling/SIM_Baseline_Validation_Predictions.csv"

# Load baseline predictions
df_baseline = pd.read_csv(
    baseline_file,
    parse_dates=["Date"]
)

# Inspect the baseline dataset
print("Baseline dataset dimensions:", df_baseline.shape)

print("\nBaseline columns:")
print(df_baseline.columns.tolist())

print("\nDate range:")
print("Start:", df_baseline["Date"].min().date())
print("End:", df_baseline["Date"].max().date())

print("\nFirst five observations:")
display(df_baseline.head())


Baseline dataset dimensions: (8760, 8)

Baseline columns:
['Date', 'DIVISION', 'Window', 'Baseline_Prediction', 'Intervention_Count', 'Error', 'Absolute_Error', 'Year']

Date range:
Start: 2021-01-01
End: 2024-12-30

First five observations:


,Date,DIVISION,Window,Baseline_Prediction,Intervention_Count,Error,Absolute_Error,Year
0,2021-01-01,1,13,22.384615,18.0,4.384615,4.384615,2021
1,2021-01-01,2,13,51.230769,41.0,10.230769,10.230769,2021
2,2021-01-01,3,13,52.923077,48.0,4.923077,4.923077,2021
3,2021-01-01,4,13,58.461538,53.0,5.461538,5.461538,2021
4,2021-01-01,5,13,53.769231,42.0,11.769231,11.769231,2021


## 5.11. Match LightGBM and Baseline Predictions

We will match the LightGBM predictions with the selected baseline predictions using the forecast date and operational division.

The comparison will use only observations available for both forecasting methods.

We will verify that each date-division combination is unique, that all validation observations are matched, and that both datasets contain the same actual intervention counts.

This verification is necessary before comparing MAE and prediction bias across forecasting methods.

In [ ]:

# Check for duplicate date-division combinations
lightgbm_duplicates = df_lightgbm_predictions.duplicated(
    subset=["Date", "DIVISION"]
).sum()

baseline_duplicates = df_baseline.duplicated(
    subset=["Date", "DIVISION"]
).sum()

print("LightGBM duplicates:", lightgbm_duplicates)
print("Baseline duplicates:", baseline_duplicates)

# Select only the columns needed for comparison
lightgbm_results = df_lightgbm_predictions[
    ["Date", "DIVISION", "Intervention_Count", "Predicted_Count"]
].copy()

baseline_results = df_baseline[
    ["Date", "DIVISION", "Intervention_Count", "Baseline_Prediction"]
].copy()

# Match predictions using date and division
df_comparison = pd.merge(
    lightgbm_results,
    baseline_results,
    on=["Date", "DIVISION"],
    how="inner",
    suffixes=("_LightGBM", "_Baseline"),
    validate="one_to_one"
)

# Verify that actual intervention counts match
actual_counts_match = (
    df_comparison["Intervention_Count_LightGBM"] ==
    df_comparison["Intervention_Count_Baseline"]
).all()

print("\nLightGBM observations:", len(lightgbm_results))
print("Baseline observations:", len(baseline_results))
print("Matched observations:", len(df_comparison))
print("Actual intervention counts match:", actual_counts_match)

# Display the first matched predictions
display(df_comparison.head())


LightGBM duplicates: 0
Baseline duplicates: 0

LightGBM observations: 8760
Baseline observations: 8760
Matched observations: 8760
Actual intervention counts match: True


,Date,DIVISION,Intervention_Count_LightGBM,Predicted_Count,Intervention_Count_Baseline,Baseline_Prediction
0,2021-01-01,1,18.0,23.121406,18.0,22.384615
1,2021-01-01,2,41.0,54.634131,41.0,51.230769
2,2021-01-01,3,48.0,58.526979,48.0,52.923077
3,2021-01-01,4,53.0,58.386332,53.0,58.461538
4,2021-01-01,5,42.0,58.798730,42.0,53.769231


**Methodological Observation**

LightGBM and baseline predictions were successfully matched using forecast date and operational division.

Both datasets contained 8,760 validation observations, with no duplicate date-division combinations. All observations were matched, and the actual intervention counts were identical across both datasets.

This confirms that the two forecasting methods can be compared using the same validation observations from 2021 through 2024.

## 5.12. Compare LightGBM and Baseline Performance

We will compare LightGBM with the selected 13-observation same-weekday baseline using the same 8,760 validation observations.

For each method, we will calculate Mean Absolute Error (MAE) and prediction bias.

We will also calculate the absolute and percentage improvement in MAE to assess whether LightGBM provides a measurable forecasting benefit over the simpler baseline.

The comparison will be performed overall and separately for each operational division. The reserved final test period will remain excluded.

In [ ]:

# Use the verified actual intervention counts
df_comparison["Actual_Count"] = (
    df_comparison["Intervention_Count_LightGBM"]
)

# Calculate the prediction errors for both methods
df_comparison["LightGBM_Error"] = (
    df_comparison["Predicted_Count"] -
    df_comparison["Actual_Count"]
)

df_comparison["Baseline_Error"] = (
    df_comparison["Baseline_Prediction"] -
    df_comparison["Actual_Count"]
)

# Calculate overall MAE
lightgbm_mae = df_comparison["LightGBM_Error"].abs().mean()
baseline_mae = df_comparison["Baseline_Error"].abs().mean()

# Calculate overall prediction bias
lightgbm_bias = df_comparison["LightGBM_Error"].mean()
baseline_bias = df_comparison["Baseline_Error"].mean()

# Calculate MAE improvement
mae_improvement = baseline_mae - lightgbm_mae
improvement_percent = (mae_improvement / baseline_mae) * 100

# Display the overall results
print("Overall Model Comparison (2021-2024)")
print("Baseline MAE:", round(baseline_mae, 3))
print("LightGBM MAE:", round(lightgbm_mae, 3))
print("MAE Improvement:", round(mae_improvement, 3))
print("MAE Improvement (%):", round(improvement_percent, 2))

print("\nBaseline Bias:", round(baseline_bias, 3))
print("LightGBM Bias:", round(lightgbm_bias, 3))

# Calculate MAE and bias by division
division_comparison = df_comparison.groupby("DIVISION").agg(
    Baseline_MAE=(
        "Baseline_Error",
        lambda errors: errors.abs().mean()
    ),
    LightGBM_MAE=(
        "LightGBM_Error",
        lambda errors: errors.abs().mean()
    ),
    Baseline_Bias=("Baseline_Error", "mean"),
    LightGBM_Bias=("LightGBM_Error", "mean")
)

# Calculate improvement for each division
division_comparison["MAE_Improvement"] = (
    division_comparison["Baseline_MAE"] -
    division_comparison["LightGBM_MAE"]
)

print("\nComparison by division:")
display(division_comparison.round(3))


Overall Model Comparison (2021-2024)
Baseline MAE: 8.48
LightGBM MAE: 8.15
MAE Improvement: 0.33
MAE Improvement (%): 3.89

Baseline Bias: -0.483
LightGBM Bias: -1.913

Comparison by division:


,Baseline_MAE,LightGBM_MAE,Baseline_Bias,LightGBM_Bias,MAE_Improvement
DIVISION,,,,,
1,6.319,6.035,-0.337,-1.351,0.284
2,8.773,8.255,-0.443,-1.496,0.518
3,9.707,9.442,-0.597,-3.407,0.265
4,8.844,8.679,-0.372,-2.200,0.165
5,9.359,8.846,-0.365,-1.798,0.513
6,7.880,7.644,-0.783,-1.227,0.236


**Methodological Observation**

LightGBM and the selected 13-observation same-weekday baseline were compared using the same 8,760 validation observations from 2021 through 2024.

LightGBM achieved an overall MAE of 8.150, compared with 8.480 for the baseline. This represents a reduction of 0.330 interventions, or approximately 3.89%.

LightGBM achieved a lower MAE in all six operational divisions. However, its prediction bias was more negative (−1.913) than the baseline bias (−0.483), indicating a stronger tendency to underestimate intervention counts.

These findings demonstrate a modest improvement in average absolute prediction accuracy, accompanied by greater systematic underestimation. The results do not yet establish whether the additional model complexity is justified.

## 5.13. Compare Model Performance by Year

We will compare LightGBM and the selected baseline separately for each validation year from 2021 through 2024.

This analysis will help determine whether the overall MAE improvement is consistent across years or concentrated in particular periods.

We will calculate annual MAE, prediction bias, and the absolute improvement in MAE.

The results will support a more balanced assessment of forecasting accuracy and temporal stability before introducing CatBoost.

In [ ]:

# Extract the year from the forecast date
df_comparison["Year"] = df_comparison["Date"].dt.year

# Calculate annual performance for both methods
yearly_comparison = df_comparison.groupby("Year").agg(
    Baseline_MAE=(
        "Baseline_Error",
        lambda errors: errors.abs().mean()
    ),
    LightGBM_MAE=(
        "LightGBM_Error",
        lambda errors: errors.abs().mean()
    ),
    Baseline_Bias=("Baseline_Error", "mean"),
    LightGBM_Bias=("LightGBM_Error", "mean")
)

# Calculate the annual MAE improvement
yearly_comparison["MAE_Improvement"] = (
    yearly_comparison["Baseline_MAE"] -
    yearly_comparison["LightGBM_MAE"]
)

# Display the comparison
print("Annual Model Comparison (2021-2024)")
display(yearly_comparison.round(3))


Annual Model Comparison (2021-2024)


,Baseline_MAE,LightGBM_MAE,Baseline_Bias,LightGBM_Bias,MAE_Improvement
Year,,,,,
2021,7.255,7.191,-0.245,-0.778,0.064
2022,7.789,7.671,-1.187,-2.370,0.118
2023,10.719,9.784,-0.111,-3.530,0.935
2024,8.159,7.954,-0.387,-0.974,0.205


**Methodological Observation**

The annual comparison shows that LightGBM achieved a lower MAE than the selected baseline in each validation year from 2021 through 2024.

The largest improvement occurred in 2023, with an MAE reduction of 0.935 interventions. However, 2023 remained the year with the highest forecasting errors for both methods.

The improvement was relatively small in 2021 (0.064 interventions), indicating that the benefit of LightGBM varied across validation years.

LightGBM also showed a stronger negative prediction bias in every year, particularly in 2023. Therefore, although the model consistently reduced MAE, its greater tendency to underestimate intervention counts remains an important limitation.

These findings support evaluating additional models before deciding whether the increased modeling complexity is justified.

## 5.14. Install and Import CatBoost

We will introduce CatBoost as the second machine learning model for forecasting daily SIM intervention counts.

Like LightGBM, CatBoost uses gradient-boosted decision trees. One important difference is its ability to handle categorical predictors directly.

We will treat `DIVISION` as a categorical feature because operational division identifiers represent distinct categories rather than numerical quantities.

CatBoost will be evaluated using the same chronological validation schedule and target variable as LightGBM.

In [ ]:
# LightGBM and CatBoost are project dependencies (pyproject.toml): no install needed in the repo.

In [ ]:

# Import CatBoost
from catboost import CatBoostRegressor

# Verify the installation
import catboost

print("CatBoost version:", catboost.__version__)


CatBoost version: 1.2.10


## 5.15. Prepare Categorical Features for CatBoost

We will prepare the predictive features for the initial CatBoost model.

The `DIVISION` column represents six operational categories rather than a continuous numerical quantity. We will convert this column to a string and identify it as a categorical feature for CatBoost.

The remaining predictive features will retain their numerical values, including any missing historical values.

We will use the same January 2021 training and validation observations previously used for the initial LightGBM model to maintain a consistent comparison.

In [ ]:

# Create separate feature datasets for CatBoost
X_train_cat = X_train.copy()
X_val_cat = X_val.copy()

# Convert DIVISION into a categorical identifier
X_train_cat["DIVISION"] = X_train_cat["DIVISION"].astype(str)
X_val_cat["DIVISION"] = X_val_cat["DIVISION"].astype(str)

# Identify the categorical feature
categorical_features = ["DIVISION"]

# Verify the categorical values
print("Categorical features:", categorical_features)

print(
    "Training divisions:",
    sorted(X_train_cat["DIVISION"].unique())
)

print(
    "Validation divisions:",
    sorted(X_val_cat["DIVISION"].unique())
)

# Verify dataset dimensions
print("\nTraining features:", X_train_cat.shape)
print("Validation features:", X_val_cat.shape)

# Verify missing numerical values
print(
    "Missing training feature values:",
    X_train_cat.isna().sum().sum()
)

print(
    "Missing validation feature values:",
    X_val_cat.isna().sum().sum()
)


Categorical features: ['DIVISION']
Training divisions: ['1', '2', '3', '4', '5', '6']
Validation divisions: ['1', '2', '3', '4', '5', '6']

Training features: (2184, 12)
Validation features: (186, 12)
Missing training feature values: 462
Missing validation feature values: 0


## 5.16. Train the First CatBoost Model

We will train an initial CatBoost regression model using historical observations available through December 29, 2020.

The model will use the same 12 candidate predictors and target variable as LightGBM, with `DIVISION` explicitly treated as categorical.

We will begin with a simple configuration without hyperparameter tuning.

The trained model will generate predictions for January 2021. These predictions will allow us to evaluate its initial performance before applying the complete monthly validation schedule.

In [ ]:

# Create the CatBoost regression model
catboost_model = CatBoostRegressor(
    iterations=100,
    learning_rate=0.05,
    depth=4,
    loss_function="RMSE",
    random_seed=42,
    verbose=False,
    thread_count=2
)

# Train the model
catboost_model.fit(
    X_train_cat,
    y_train,
    cat_features=categorical_features
)

# Generate predictions for January 2021
catboost_predictions = catboost_model.predict(X_val_cat)

# Prepare the prediction results
df_catboost_first = df_val[
    ["Date", "DIVISION", "Intervention_Count"]
].copy()

df_catboost_first["Predicted_Count"] = catboost_predictions

# Display results
print("CatBoost training completed.")
print("Number of predictions:", len(catboost_predictions))

display(df_catboost_first.head(10))


CatBoost training completed.
Number of predictions: 186


,Date,DIVISION,Intervention_Count,Predicted_Count
2196,2021-01-01,1,18.0,25.721262
2197,2021-01-01,2,41.0,54.572590
2198,2021-01-01,3,48.0,53.516553
2199,2021-01-01,4,53.0,56.256852
2200,2021-01-01,5,42.0,56.173112
2201,2021-01-01,6,27.0,28.867867
2202,2021-01-02,1,22.0,27.957362
2203,2021-01-02,2,35.0,52.738028
2204,2021-01-02,3,48.0,54.030654
2205,2021-01-02,4,48.0,56.653569


## 5.17. Evaluate the First CatBoost Model

We will evaluate the initial CatBoost model using the 186 validation observations from January 2021.

Mean Absolute Error (MAE) will measure the average magnitude of prediction errors, while prediction bias will indicate whether the model tends to overestimate or underestimate intervention counts.

We will also calculate MAE separately for each operational division.

These results provide an initial assessment of CatBoost performance. The complete 2021–2024 monthly validation will be required before comparing the models across the full development period.

In [ ]:

# Calculate prediction errors
df_catboost_first["Error"] = (
    df_catboost_first["Predicted_Count"] -
    df_catboost_first["Intervention_Count"]
)

# Calculate overall MAE
catboost_mae = mean_absolute_error(
    df_catboost_first["Intervention_Count"],
    df_catboost_first["Predicted_Count"]
)

# Calculate overall prediction bias
catboost_bias = df_catboost_first["Error"].mean()

# Display the results
print("CatBoost Validation Results (January 2021)")
print("Total predictions:", len(df_catboost_first))
print("MAE:", round(catboost_mae, 3))
print("Bias:", round(catboost_bias, 3))

# Calculate MAE for each division
catboost_division_mae = (
    df_catboost_first.groupby("DIVISION")["Error"]
    .apply(lambda errors: errors.abs().mean())
)

print("\nMAE by division:")
display(catboost_division_mae.round(3))


CatBoost Validation Results (January 2021)
Total predictions: 186
MAE: 7.556
Bias: 3.396

MAE by division:


DIVISION
1     6.872
2     8.040
3     5.878
4     8.197
5    12.015
6     4.334
Name: Error, dtype: float64

**Methodological Observation**

The initial CatBoost model was evaluated on 186 observations from January 2021, using the same training and validation periods as the initial LightGBM model.

CatBoost achieved an MAE of 7.556 interventions and a positive prediction bias of 3.396, indicating an average tendency to overestimate intervention counts.

Its MAE was slightly lower than LightGBM's MAE of 7.781 for the same month. CatBoost achieved lower MAE in four of the six operational divisions, while LightGBM performed better in Divisions 1 and 2.

These results provide an initial comparison only. The complete 48-month chronological validation is required to assess overall accuracy and temporal consistency.

## 5.18. Train CatBoost Across Monthly Validation Windows

We will train CatBoost models using the 48 monthly chronological validation windows from January 2021 through December 2024.

For each month, a new model will be trained using observations available through the corresponding D−3 training cutoff.

The model will remain fixed within each validation month. Daily predictive features may incorporate newly available historical intervention counts, respecting the D−3 information restriction.

The same 12 candidate predictive features and initial CatBoost configuration will be used throughout the validation period. `DIVISION` will be treated as a categorical predictor.

All validation predictions will be stored for subsequent comparison with LightGBM and the selected baseline.

In [ ]:

# Store predictions from all validation months
all_catboost_predictions = []

# Train and evaluate a new model for each month
for _, window in df_schedule.iterrows():

    # Select training observations
    monthly_train = df_modeling[
        df_modeling["Date"] <= window["Training_Cutoff"]
    ]

    # Select validation observations
    monthly_val = df_modeling[
        (df_modeling["Date"] >= window["Validation_Start"]) &
        (df_modeling["Date"] <= window["Validation_End"])
    ]

    # Prepare the training and validation features
    X_train_month = monthly_train[feature_columns].copy()
    X_val_month = monthly_val[feature_columns].copy()

    # Convert DIVISION to a categorical identifier
    X_train_month["DIVISION"] = X_train_month["DIVISION"].astype(str)
    X_val_month["DIVISION"] = X_val_month["DIVISION"].astype(str)

    # Prepare the training target
    y_train_month = monthly_train[target_column]

    # Create a new CatBoost model
    monthly_catboost_model = CatBoostRegressor(
        iterations=100,
        learning_rate=0.05,
        depth=4,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=2
    )

    # Train the model
    monthly_catboost_model.fit(
        X_train_month,
        y_train_month,
        cat_features=categorical_features
    )

    # Generate predictions for the validation month
    monthly_predictions = monthly_catboost_model.predict(X_val_month)

    # Store forecast dates, divisions, actuals, and predictions
    monthly_results = monthly_val[
        ["Date", "DIVISION", "Intervention_Count"]
    ].copy()

    monthly_results["Predicted_Count"] = monthly_predictions

    all_catboost_predictions.append(monthly_results)

# Combine predictions from all validation months
df_catboost_predictions = pd.concat(
    all_catboost_predictions,
    ignore_index=True
)

# Verify the results
print("Validation months processed:", len(all_catboost_predictions))
print("Total predictions:", len(df_catboost_predictions))
print(
    "Missing predictions:",
    df_catboost_predictions["Predicted_Count"].isna().sum()
)

print("\nFirst five predictions:")
display(df_catboost_predictions.head())


Validation months processed: 48
Total predictions: 8760
Missing predictions: 0

First five predictions:


,Date,DIVISION,Intervention_Count,Predicted_Count
0,2021-01-01,1,18.0,25.721262
1,2021-01-01,2,41.0,54.572590
2,2021-01-01,3,48.0,53.516553
3,2021-01-01,4,53.0,56.256852
4,2021-01-01,5,42.0,56.173112


## 5.19. Evaluate CatBoost Validation Performance

We will evaluate CatBoost predictions across the 48 monthly validation windows covering 2021 through 2024.

Mean Absolute Error (MAE) will measure the average magnitude of prediction errors. Prediction bias will indicate whether CatBoost tends to overestimate or underestimate daily intervention counts.

We will also examine performance separately by operational division and calendar year to identify differences in forecasting accuracy.

These results will be used to compare CatBoost with LightGBM and the selected same-weekday baseline. The reserved final test period will remain excluded.

In [ ]:

# Calculate prediction errors
df_catboost_predictions["Error"] = (
    df_catboost_predictions["Predicted_Count"] -
    df_catboost_predictions["Intervention_Count"]
)

# Calculate overall validation metrics
catboost_overall_mae = (
    df_catboost_predictions["Error"].abs().mean()
)

catboost_overall_bias = (
    df_catboost_predictions["Error"].mean()
)

# Display overall results
print("CatBoost Validation Results (2021-2024)")
print("Total predictions:", len(df_catboost_predictions))
print("Overall MAE:", round(catboost_overall_mae, 3))
print("Overall Bias:", round(catboost_overall_bias, 3))

# Calculate performance by division
catboost_division_metrics = (
    df_catboost_predictions.groupby("DIVISION")["Error"]
    .agg(
        MAE=lambda errors: errors.abs().mean(),
        Bias="mean"
    )
)

print("\nPerformance by division:")
display(catboost_division_metrics.round(3))

# Extract the year from each forecast date
df_catboost_predictions["Year"] = (
    df_catboost_predictions["Date"].dt.year
)

# Calculate performance by year
catboost_yearly_metrics = (
    df_catboost_predictions.groupby("Year")["Error"]
    .agg(
        MAE=lambda errors: errors.abs().mean(),
        Bias="mean"
    )
)

print("\nPerformance by year:")
display(catboost_yearly_metrics.round(3))


CatBoost Validation Results (2021-2024)
Total predictions: 8760
Overall MAE: 8.127
Overall Bias: -1.768

Performance by division:


,MAE,Bias
DIVISION,,
1,6.134,-0.892
2,8.143,-0.867
3,9.322,-3.806
4,8.506,-2.565
5,8.789,-1.946
6,7.869,-0.530



Performance by year:


,MAE,Bias
Year,,
2021,7.123,-0.846
2022,7.550,-1.945
2023,9.958,-3.184
2024,7.878,-1.096


**Methodological Observation**

CatBoost was evaluated across 48 monthly validation windows covering 8,760 division-day observations from 2021 through 2024.

The model achieved an overall MAE of 8.127 interventions and a prediction bias of −1.768, indicating a general tendency to underestimate intervention counts.

CatBoost achieved a slightly lower overall MAE than LightGBM (8.150) and the selected baseline (8.480). However, its improvement over LightGBM was only 0.023 interventions.

CatBoost achieved lower MAE than LightGBM in four of the six operational divisions and in three of the four validation years. LightGBM performed better in Divisions 1 and 6 and during 2023.

Both machine learning models showed greater systematic underestimation than the baseline. Therefore, the small difference in overall MAE is not sufficient on its own to justify selecting CatBoost as the preferred model.

## 5.20. Match Baseline, LightGBM, and CatBoost Predictions

We will combine the validation predictions from the baseline, LightGBM, and CatBoost using forecast date and operational division.

Before comparing the three methods, we will verify that each date-division combination is unique, that all observations are matched, and that actual intervention counts agree across the datasets.

This will ensure that the final model comparison uses exactly the same observations from the 2021–2024 validation period.

In [ ]:

# Select CatBoost prediction columns
catboost_results = df_catboost_predictions[
    ["Date", "DIVISION", "Intervention_Count", "Predicted_Count"]
].copy()

# Rename columns to identify CatBoost values
catboost_results = catboost_results.rename(
    columns={
        "Intervention_Count": "CatBoost_Actual",
        "Predicted_Count": "CatBoost_Prediction"
    }
)

# Verify that CatBoost observations are unique
catboost_duplicates = catboost_results.duplicated(
    subset=["Date", "DIVISION"]
).sum()

print("CatBoost duplicates:", catboost_duplicates)

# Match CatBoost with the previously verified comparison
df_all_models = pd.merge(
    df_comparison,
    catboost_results,
    on=["Date", "DIVISION"],
    how="inner",
    validate="one_to_one"
)

# Verify matching actual intervention counts
actual_counts_match = (
    df_all_models["Actual_Count"] ==
    df_all_models["CatBoost_Actual"]
).all()

# Display verification results
print("Baseline and LightGBM observations:", len(df_comparison))
print("CatBoost observations:", len(catboost_results))
print("Matched observations:", len(df_all_models))
print("Actual intervention counts match:", actual_counts_match)

# Preview the matched predictions
display(
    df_all_models[
        [
            "Date",
            "DIVISION",
            "Actual_Count",
            "Baseline_Prediction",
            "Predicted_Count",
            "CatBoost_Prediction"
        ]
    ].head()
)


CatBoost duplicates: 0
Baseline and LightGBM observations: 8760
CatBoost observations: 8760
Matched observations: 8760
Actual intervention counts match: True


,Date,DIVISION,Actual_Count,Baseline_Prediction,Predicted_Count,CatBoost_Prediction
0,2021-01-01,1,18.0,22.384615,23.121406,25.721262
1,2021-01-01,2,41.0,51.230769,54.634131,54.572590
2,2021-01-01,3,48.0,52.923077,58.526979,53.516553
3,2021-01-01,4,53.0,58.461538,58.386332,56.256852
4,2021-01-01,5,42.0,53.769231,58.798730,56.173112


**Methodological Observation**

The baseline, LightGBM, and CatBoost predictions were successfully matched using forecast date and operational division.

All three methods provided predictions for the same 8,760 division-day observations from 2021 through 2024. No duplicate date-division combinations were identified, and actual intervention counts were consistent across the matched datasets.

This confirms that the three forecasting methods can be compared using identical validation observations.

## 5.21. Compare All Forecasting Methods

We will compare the selected same-weekday baseline, LightGBM, and CatBoost using the same 8,760 validation observations from 2021 through 2024.

For each forecasting method, we will calculate Mean Absolute Error (MAE) and prediction bias.

We will also calculate the percentage reduction in MAE relative to the baseline to measure the improvement provided by the machine learning models.

The results will support the assessment of predictive accuracy and model complexity before considering an ensemble approach.

In [ ]:

# Define the prediction column for each method
model_columns = {
    "Baseline": "Baseline_Prediction",
    "LightGBM": "Predicted_Count",
    "CatBoost": "CatBoost_Prediction"
}

# Store the comparison results
model_results = []

# Calculate performance for each method
for model_name, prediction_column in model_columns.items():

    # Calculate prediction errors
    errors = (
        df_all_models[prediction_column] -
        df_all_models["Actual_Count"]
    )

    # Calculate MAE and Bias
    model_mae = errors.abs().mean()
    model_bias = errors.mean()

    # Store the results
    model_results.append({
        "Model": model_name,
        "MAE": model_mae,
        "Bias": model_bias
    })

# Create the comparison table
df_model_metrics = pd.DataFrame(model_results)

# Retrieve baseline MAE
baseline_reference_mae = df_model_metrics.loc[
    df_model_metrics["Model"] == "Baseline",
    "MAE"
].iloc[0]

# Calculate percentage improvement over baseline
df_model_metrics["MAE_Improvement_Percent"] = (
    (baseline_reference_mae - df_model_metrics["MAE"]) /
    baseline_reference_mae * 100
)

# Display the final comparison
print("Forecasting Model Comparison (2021-2024)")
display(df_model_metrics.round(3))


Forecasting Model Comparison (2021-2024)


,Model,MAE,Bias,MAE_Improvement_Percent
0,Baseline,8.480,-0.483,0.000
1,LightGBM,8.150,-1.913,3.894
2,CatBoost,8.127,-1.768,4.164


**Methodological Observation**

The consolidated comparison confirmed the performance of all three forecasting methods using the same 8,760 validation observations from 2021 through 2024.

The selected baseline achieved an MAE of 8.480, LightGBM achieved 8.150, and CatBoost achieved 8.127.

Relative to the baseline, LightGBM reduced MAE by 3.894%, while CatBoost reduced it by 4.164%. However, the difference between LightGBM and CatBoost was only 0.023 interventions.

Both machine learning models showed greater negative prediction bias than the baseline, indicating a stronger tendency to underestimate intervention counts.

These results demonstrate modest improvements in average absolute accuracy but do not yet justify selecting a more complex forecasting approach. Further evaluation is required before making a final model selection.

## 5.22. Save Model Training Outputs

We will save the matched validation predictions and consolidated performance metrics from the baseline, LightGBM, and CatBoost models.

These outputs will preserve the results from the 2021–2024 chronological validation period and support subsequent model comparison and ensemble evaluation.

The saved predictions will include forecast date, operational division, actual intervention count, and predictions from each method.

The reserved 2025–2026 final test observations will not be included in these files.

In [ ]:

# Select and rename the columns for the final prediction file
df_validation_predictions = df_all_models[
    [
        "Date",
        "DIVISION",
        "Actual_Count",
        "Baseline_Prediction",
        "Predicted_Count",
        "CatBoost_Prediction"
    ]
].copy()

df_validation_predictions = df_validation_predictions.rename(
    columns={
        "Predicted_Count": "LightGBM_Prediction"
    }
)

# Define output file paths
predictions_file = "../../data/interim/modelling/SIM_Model_Validation_Predictions.csv"
metrics_file = "../../data/interim/modelling/SIM_Model_Validation_Metrics.csv"

# Save validation predictions
df_validation_predictions.to_csv(
    predictions_file,
    index=False
)

# Save consolidated metrics
df_model_metrics.to_csv(
    metrics_file,
    index=False
)

# Confirm saved results
print("Validation predictions saved:", predictions_file)
print("Validation metrics saved:", metrics_file)

print("\nPrediction records:", len(df_validation_predictions))
print("Models evaluated:", len(df_model_metrics))

# Verify that the reserved test period is excluded
print(
    "Final test period excluded:",
    df_validation_predictions["Date"].max() < pd.Timestamp("2025-01-01")
)

# Preview saved data
display(df_validation_predictions.head())


Validation predictions saved: ../../data/interim/modelling/SIM_Model_Validation_Predictions.csv
Validation metrics saved: ../../data/interim/modelling/SIM_Model_Validation_Metrics.csv

Prediction records: 8760
Models evaluated: 3
Final test period excluded: True


,Date,DIVISION,Actual_Count,Baseline_Prediction,LightGBM_Prediction,CatBoost_Prediction
0,2021-01-01,1,18.0,22.384615,23.121406,25.721262
1,2021-01-01,2,41.0,51.230769,54.634131,54.572590
2,2021-01-01,3,48.0,52.923077,58.526979,53.516553
3,2021-01-01,4,53.0,58.461538,58.386332,56.256852
4,2021-01-01,5,42.0,53.769231,58.798730,56.173112


## 5.23. Stage Conclusion — Model Training

This stage trained and evaluated LightGBM and CatBoost models for forecasting daily SIM intervention workloads across six operational divisions.

Both models used 12 candidate predictive features and were evaluated using 48 monthly expanding-window validation periods from 2021 through 2024. Training data were restricted to the corresponding D−3 cutoff, and each model remained fixed during its validation month.

The evaluation included 8,760 matched division-day observations and produced the following results:

| Model | MAE | Bias | MAE Improvement vs. Baseline |
|---|---:|---:|---:|
| Same-weekday baseline | 8.480 | −0.483 | 0.000% |
| LightGBM | 8.150 | −1.913 | 3.894% |
| CatBoost | 8.127 | −1.768 | 4.164% |

Both machine learning models achieved lower MAE than the selected baseline across all six operational divisions. CatBoost produced the lowest overall MAE, although its advantage over LightGBM was only 0.023 interventions per division-day.

Both machine learning models also demonstrated greater negative prediction bias than the baseline, indicating stronger systematic underestimation.

The validation predictions and consolidated metrics were saved as `SIM_Model_Validation_Predictions.csv` and `SIM_Model_Validation_Metrics.csv`.

**Conclusion:** LightGBM and CatBoost demonstrated modest improvements in average absolute forecasting accuracy relative to the baseline. However, their greater negative bias and the small difference between their MAE values do not yet justify selecting a more complex forecasting approach. Additional evaluation, including Exponential Smoothing and potential ensemble configurations, is required before final model selection.

The reserved 2025–2026 test period remains unevaluated.